In [20]:
import pandas as pd
import numpy as np
import math
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, ParameterSampler
import joblib

In [2]:
data = pd.read_csv("hour.csv")

In [3]:
data.head(4)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13


- instant: record index
- dteday : date
- season : season (1:springer, 2:summer, 3:fall, 4:winter)
- yr : year (0: 2011, 1:2012)
- mnth : month ( 1 to 12)
- hr : hour (0 to 23)
- holiday : weather day is holiday or not (extracted from http://dchr.dc.gov/page/holiday-schedule)
- weekday : day of the week
- workingday : if day is neither weekend nor holiday is 1, otherwise is 0.
+ weathersit : 
   - 1: Clear, Few clouds, Partly cloudy, Partly cloudy
   - 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
   - 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
   - 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog
- temp : Normalized temperature in Celsius. The values are divided to 41 (max)
- atemp: Normalized feeling temperature in Celsius. The values are divided to 50 (max)
- hum: Normalized humidity. The values are divided to 100 (max)
- windspeed: Normalized wind speed. The values are divided to 67 (max)
- casual: count of casual users
- registered: count of registered users
- cnt: count of total rental bikes including both casual and registered

- There are no missing values in the data. However, the column names could be more readable

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


In [5]:
X = data.drop(columns=["cnt", "casual", "registered", "instant", "dteday", "atemp"])
y = data["cnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
class BikeDataPreprocessor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self  # transforming

    def transform(self, X):
        X = X.copy()

        # renames
        X.rename(
            columns={
                "yr": "year",
                "mnth": "month",
                "holiday": "is_holiday",
                "workingday": "is_workingday",
                "weathersit": "weather_conditions",
                "hum": "humidity",
                "hr": "hour",
            },
            inplace=True,
        )

        return X

In [7]:
data.head(1)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16


In [8]:
# column groups
categorical_cols = [
    "season",
    "year",
    "month",
    "hour",
    "is_holiday",
    "is_workingday",
    "weather_conditions",
    "weekday",
]

numerical_cols = ["temp", "humidity", "windspeed"]

### Why OrdinalEncoder

Models like **XGBoost** and **Random Forest** are tree-based.  
They split data using thresholds such as `hr < 12` or `season >= 3`.

These models do **not rely on numerical order meaningfully**.  
They simply find the best split point in the data.

Example:
- `hr` encoded as `0–23` works fine
- The tree can learn that `hr = 8` and `hr = 17` both correspond to high demand.

---

### When OneHotEncoder Is Needed

For models like **Linear Regression** or **SVM**, ordinal encoding can be misleading.

These models treat numbers as having **mathematical order and distance**.

Example:
- `season = 4` would be interpreted as **greater than** `season = 1`
- But seasons have **no true numeric relationship**

In such cases, **OneHotEncoder** should be used to represent categories properly.

In [ ]:
# Numerical: nulls + encode
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),  # for NaN
        ("scaler", StandardScaler()),
    ]
)

# Categorical: nulls + encode
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),  # for NaN
        
        (    "encoder",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

In [ ]:
# combining both of them
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

full_pipeline = Pipeline(
    steps=[
        ("rename_fix", BikeDataPreprocessor()),  # 1: clean raw column names
        ("preprocess", preprocessor),  # 2: impute + scale + encode
        (
            "model",
            RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42),
        ),  # 3: model
    ]
)

# fit BikeDataPreprocessor -> ColumnTransformer -> RandomForest
full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_test)
rmse = math.sqrt(mean_squared_error(y_test, predictions))
print(f"RMSE: {rmse:.4f}")

RMSE: 42.5895


In [11]:
# cv scores
cv_scores = cross_val_score(
    full_pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error"
)
print(f"CV RMSE: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV RMSE: 66.4856 ± 7.3185


In [13]:
# param_grid = {
#     'model__n_estimators': [100, 200, 300],
#     'model__max_depth': [10, 15, 20, None],
#     'model__min_samples_split': [2, 5, 10]
# }

# search = RandomizedSearchCV(full_pipeline, param_grid, cv=5,
#                             scoring='neg_root_mean_squared_error',
#                             n_iter=20, random_state=42)
# search.fit(X_train, y_train)

# print("Best Params:", search.best_params_)
# print("Best RMSE:", -search.best_score_)

In [19]:
xgb_pipeline = Pipeline(
    steps=[
        ("rename_fix", BikeDataPreprocessor()),
        ("preprocess", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=8,
                colsample_bytree=0.8,
                subsample=0.8,
                random_state=42,
            ),
        ),
    ]
)

xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)
xgb_rmse = math.sqrt(mean_squared_error(y_test, xgb_preds))
print(f"XGB RMSE: {xgb_rmse:.4f}")

XGB RMSE: 37.3513


In [21]:
joblib.dump(xgb_pipeline, "bike_pipeline_final.pkl")

['bike_pipeline_final.pkl']

In [25]:
loaded_pipeline = joblib.load("bike_pipeline_final.pkl")
preds = loaded_pipeline.predict(X_test)
print(preds[:5])

comparison = pd.DataFrame(
    {"Actual": y_test.values[:5], "Predicted": preds[:5].astype(int)}
)
print(comparison)

[398.60135  101.35281    6.603586 538.9021    21.197897]
   Actual  Predicted
0     425        398
1      88        101
2       4          6
3     526        538
4      13         21


In [26]:
new_data = pd.DataFrame(
    [
        {
            "season": 1,
            "yr": 0,
            "mnth": 1,
            "hr": 8,
            "holiday": 0,
            "weekday": 1,
            "workingday": 1,
            "weathersit": 1,
            "temp": 0.24,
            "atemp": 0.2273,
            "hum": 0.81,
            "windspeed": 0.0,
        },
        {
            "season": 2,
            "yr": 1,
            "mnth": 6,
            "hr": 17,
            "holiday": 0,
            "weekday": 5,
            "workingday": 1,
            "weathersit": 2,
            "temp": 0.60,
            "atemp": 0.5758,
            "hum": 0.65,
            "windspeed": 0.2,
        },
    ]
)

predictions = loaded_pipeline.predict(new_data)
for i, pred in enumerate(predictions):
    print(f"Row {i+1}: {int(pred)} bikes")

Row 1: 202 bikes
Row 2: 706 bikes


In [30]:
loaded_pipeline = joblib.load("bike_pipeline_final.pkl")
new_data = pd.read_csv("test.csv")
predictions = loaded_pipeline.predict(new_data)

# results table
results = new_data[["season", "hr", "weekday", "weathersit", "temp"]].copy()
results["predicted_bikes"] = predictions.astype(int)

# changing labels
results["season_label"] = results["season"].map(
    {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"}
)
results["time"] = results["hr"].apply(lambda x: f"{x:02d}:00")
results["weather_label"] = results["weathersit"].map(
    {1: "Clear", 2: "Cloudy", 3: "Rain/Snow"}
)
results["weekday_label"] = results["weekday"].map(
    {0: "Sun", 1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat"}
)

# Final
final = results[
    [
        "season_label",
        "time",
        "weekday_label",
        "weather_label",
        "temp",
        "predicted_bikes",
    ]
]
final.columns = ["Season", "Time", "Day", "Weather", "Temp", "Predicted Bikes"]

print(final.to_string(index=False))

Season  Time Day   Weather   Temp  Predicted Bikes
Summer 07:00 Thu     Clear 0.2248              106
  Fall 20:00 Wed Rain/Snow 0.2699              166
Spring 16:00 Tue     Clear 0.1373              154
Spring 22:00 Sun     Clear 0.7880               88
Winter 08:00 Thu     Clear 0.6474              609
Summer 03:00 Wed     Clear 0.4401                5
Spring 01:00 Fri    Cloudy 0.5783               47
Winter 20:00 Mon    Cloudy 0.5694              263
Winter 06:00 Sun    Cloudy 0.3348                4
Summer 04:00 Sat     Clear 0.1927                7
  Fall 23:00 Tue    Cloudy 0.6100              130
Spring 04:00 Sat Rain/Snow 0.4950               -3
Winter 16:00 Thu    Cloudy 0.8261              211
  Fall 18:00 Mon     Clear 0.7465              330
  Fall 22:00 Sun Rain/Snow 0.1880               51
Winter 05:00 Sun Rain/Snow 0.2777                0
Spring 08:00 Sun Rain/Snow 0.8774              141
Winter 23:00 Tue     Clear 0.8813               84
Winter 20:00 Mon Rain/Snow 0.29